In [191]:
import pandas as pd
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from nltk.stem import WordNetLemmatizer
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

In [192]:
MIN_REVIEWS = 35
TOP_K = 10
RNG = 1

# Filtering users by number of reviews done and items by number of reviews received

In [193]:
reviews = pd.read_csv("reviews_clean.csv")
meta = pd.read_csv("meta_clean.csv")

user_counts = reviews["user_id"].value_counts()
item_counts = reviews["parent_asin"].value_counts()

suitable_users = user_counts[user_counts >= MIN_REVIEWS].index
suitable_items = item_counts[item_counts >= MIN_REVIEWS].index

ratings = reviews[
    reviews["user_id"].isin(suitable_users) &
    reviews["parent_asin"].isin(suitable_items)
].copy()

ratings["rating"] = ratings["rating"].astype(float)

print(f'Min_Reviews: {MIN_REVIEWS}')
print(f'Ratings considerati: {len(ratings)}')
print(f'Utenti considerati: {ratings["user_id"].nunique()}')
print(f'Item considerati: {ratings["parent_asin"].nunique()}')


Min_Reviews: 35
Ratings considerati: 70646
Utenti considerati: 1625
Item considerati: 11217


# Filtering items with empty descriptions

In [194]:
meta = meta[meta["parent_asin"].isin(ratings["parent_asin"])].copy()

meta = meta[meta["description"].str.len() > 2].copy()
#Siccome gli item senza descrizione hanno caratteri '[' e ']', per essere considerata vuota deve avere lunghezza=2

print(f'Item considerati dopo rimozione descrizioni vuote: {len(meta)}')

Item considerati dopo rimozione descrizioni vuote: 8945


# Filtering items with short descriptions

In [195]:
meta = meta[meta["description"].str.findall(r"\b[a-zA-Z0-9]{2,}\b").str.len() >= 35].copy()

print(f'Item considerati dopo filtro descrizioni corte: {len(meta)}')

Item considerati dopo filtro descrizioni corte: 7726


# Updating reviews after removing items without description

In [196]:
valid_items = meta["parent_asin"]
ratings = ratings[ratings["parent_asin"].isin(valid_items)].copy()
len(ratings)
print(f'Ratings considerati dopo rimozione item senza descrizione: {len(ratings)}')
print(f'Utenti considerati dopo rimozione item senza descrizione: {ratings["user_id"].nunique()}')

Ratings considerati dopo rimozione item senza descrizione: 57417
Utenti considerati dopo rimozione item senza descrizione: 1622


# Merging title with description

In [197]:
meta.reset_index(inplace=True, drop=True)
meta["text"] = meta["title"] + " ; " + meta["description"]

# User x item matrix overview

In [198]:


user_item_matrix = ratings.pivot_table(
    index="user_id",
    columns="parent_asin",
    values="rating",
    aggfunc="mean"
)

user_item_matrix = user_item_matrix.reindex(columns=meta["parent_asin"])
item_titles = meta.set_index("parent_asin")["title"].reindex(user_item_matrix.columns)
avg_rating = pd.read_csv("avg_rating.csv")
item_average_ratings = avg_rating.set_index("parent_asin")["average_rating"].reindex(user_item_matrix.columns)

print(f'Dimensione matrice user x item: {user_item_matrix.shape[0]} x {user_item_matrix.shape[1]}')
display(user_item_matrix.head())


Dimensione matrice user x item: 1622 x 7726


parent_asin,B00Z9TLVK0,B00BJH85SW,B001EYUX4Y,B004BV5O0U,B001ELJEA6,B00B5QGJ5S,B09CG15F86,B001EYUWL8,B00BN5T30E,B000EP3ZLC,...,B00C1ZBFTW,B00NY5ZO0Y,B000P46NKC,B07JYW33NR,B008MAZ0PU,B09GHQVX3N,B000B8J7K0,B0716Q4932,B00XUYWTUC,B0862GHVT9
user_id,,,,,,,,,,,,,,,,,,,,,
AE25ZDXYBK3LHKCZ7XUODANPME4A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AE2A5TMJ6YE6ZNWUAFTC6P5XAHXA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AE2AZ2MNROPF33U6SS53VI22OXJA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AE2CHTOI3XIPJZLQN6DU76LI3VPQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AE2DHCNGTJRKHBHMYMCMKFDKVS2Q,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Vocab size with tf-idf Vectorizer's default tokenizer

In [199]:
vectorizer = TfidfVectorizer()
tfidf_model = vectorizer.fit_transform(meta["text"])
tfidf_item_embeddings = pd.DataFrame(tfidf_model.toarray(), columns=vectorizer.get_feature_names_out())
tfidf_item_embeddings["item_id"] = meta["parent_asin"]
display(tfidf_item_embeddings)
display(f'Vocabalari globali: {len(vectorizer.get_feature_names_out())}')

,00,000,000009,00001,00052,000599,000853,000dpi,000hz,000hzimpedance,...,❸plug,ꡧ2m,ﬁght,ﬁghting,ﬁgures,ﬁle,ﬁrst,ﬂex,ﬂuid,item_id
0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,B00Z9TLVK0
1,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,B00BJH85SW
2,0.0,0.051122,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,B001EYUX4Y
3,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,B004BV5O0U
4,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,B001ELJEA6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7721,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,B09GHQVX3N
7722,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,B000B8J7K0
7723,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,B0716Q4932
7724,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,B00XUYWTUC


'Vocabalari globali: 43798'

# Vocab size with tf-idf Vectorizer using a custom tokenizer: lemmatization, stopword removal, digits removal

In [200]:

stop_words = set(stopwords.words("english"))

TOKEN_PATTERN = re.compile(r"\b[a-zA-Z][a-zA-Z0-9]{2,}\b")

lemmatizer = WordNetLemmatizer()

def custom_tokenizer(text):
    tokens = TOKEN_PATTERN.findall(text.lower())

    return [
        lemmatizer.lemmatize(
            lemmatizer.lemmatize(
                lemmatizer.lemmatize(t, pos="n"),
                pos="v"
            ),
            pos="a"
        )
        for t in tokens
        if not t.isdigit()
        and t not in stop_words
    ]


vectorizer = TfidfVectorizer(
    tokenizer=custom_tokenizer,
    token_pattern=None
)
tfidf_model = vectorizer.fit_transform(meta["text"])

tfidf_item_embeddings = pd.DataFrame(tfidf_model.toarray(), columns=vectorizer.get_feature_names_out())

tfidf_item_embeddings.index = meta["parent_asin"]
tfidf_item_embeddings.index.name = "parent_asin"

display(tfidf_item_embeddings)
display(f'Vocabalari globali: {len(vectorizer.get_feature_names_out())}')

,a001,a10,a11,a110,a1200,a20,a2dp,a30,a30turtle,a3509,...,zumba,zumbathon,zune,zuntata,zur,zurg,zurkon,zuzong,zxr,zynaps
parent_asin,,,,,,,,,,,,,,,,,,,,,
B00Z9TLVK0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
B00BJH85SW,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
B001EYUX4Y,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
B004BV5O0U,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
B001ELJEA6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
B09GHQVX3N,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
B000B8J7K0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
B0716Q4932,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


'Vocabalari globali: 31333'

# Neural item embeddings

In [201]:
neural_model = SentenceTransformer("sentence-transformers/average_word_embeddings_komninos")
neural_vectors = neural_model.encode(meta["text"].tolist())


neural_item_embeddings = pd.DataFrame(
    neural_vectors,
    index=meta["parent_asin"],
)

neural_item_embeddings.index.name = "parent_asin"


print(f"Matrice embedding neurale item x dimensioni embedding: {neural_item_embeddings.shape[0]} x {neural_item_embeddings.shape[1]}")
display(neural_item_embeddings.head())



Matrice embedding neurale item x dimensioni embedding: 7726 x 300


,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
parent_asin,,,,,,,,,,,,,,,,,,,,,
B00Z9TLVK0,0.169324,0.076527,0.038526,-0.035707,-0.185244,-0.018477,-0.306210,0.025494,-0.140713,0.082639,...,0.318523,-0.071720,-0.158882,-0.001827,-0.277579,0.157362,0.502325,0.143491,0.316361,-0.214691
B00BJH85SW,0.112021,0.079643,-0.009534,0.019929,-0.264484,-0.095233,-0.231392,0.006419,-0.203813,0.094269,...,0.346040,0.007045,-0.101745,0.065267,-0.112259,0.212888,0.454719,0.072602,0.319091,-0.226787
B001EYUX4Y,0.189580,0.021663,0.025832,-0.036759,-0.257165,-0.045251,-0.144354,0.064650,-0.250594,0.136628,...,0.321029,0.022587,-0.094755,0.081142,-0.138593,0.256017,0.375553,0.024420,0.262903,-0.233137
B004BV5O0U,0.122890,0.095266,-0.008806,-0.059679,-0.197761,-0.044235,-0.136381,0.061547,-0.215845,0.135608,...,0.347197,-0.033369,-0.130341,0.010220,-0.095717,0.241782,0.435224,0.044573,0.296383,-0.237951
B001ELJEA6,0.070138,0.122309,-0.001762,-0.055079,-0.264183,-0.087492,-0.141680,0.029000,-0.209271,0.125484,...,0.297121,0.049294,-0.091336,0.012829,-0.063724,0.253266,0.491370,0.085002,0.274397,-0.343518


# Helper functions

In [202]:
def build_item_similarity_matrix(item_embeddings):
    item_values = np.array(item_embeddings.to_numpy(dtype=np.float32), copy=True)
    similarity_values = np.array(cosine_similarity(item_values), dtype=np.float32, copy=True)
    np.fill_diagonal(similarity_values, 0.0)

    similarity_matrix = pd.DataFrame(
        similarity_values,
        index=item_embeddings.index,
        columns=item_embeddings.index
    )
    return similarity_matrix


def predict_item_rating_knn(user_ratings, neighbors):
    rated_neighbors = user_ratings.loc[neighbors.index].dropna()

    if rated_neighbors.empty:
        fallback_ratings = item_average_ratings.loc[neighbors.index]
        return float(np.dot(fallback_ratings.values, neighbors.values) / neighbors.sum())

    weights = neighbors.loc[rated_neighbors.index]
    denominator = weights.sum()

    if denominator == 0:
        return float(rated_neighbors.mean())

    return float(np.dot(rated_neighbors.values, weights.values) / denominator)


def fill_missing_with_item_knn(base_matrix, item_similarity, top_k=10):
    filled_matrix = base_matrix.copy()
    top_neighbors = {
        item_id: item_similarity.loc[item_id].sort_values(ascending=False).head(top_k)
        for item_id in filled_matrix.columns
    }

    for user_id in filled_matrix.index:
        user_ratings = filled_matrix.loc[user_id]
        missing_items = user_ratings[user_ratings.isna()].index

        for item_id in missing_items:
            pred = predict_item_rating_knn(
                user_ratings=user_ratings,
                neighbors=top_neighbors[item_id]
            )
            filled_matrix.at[user_id, item_id] = pred

    return filled_matrix


def evaluate_item_knn(ratings_test, train_matrix, item_similarity, top_k=10):
    top_neighbors = {
        item_id: item_similarity.loc[item_id].sort_values(ascending=False).head(top_k)
        for item_id in train_matrix.columns
    }

    predictions = []
    for row in ratings_test.itertuples(index=False):
        user_ratings = train_matrix.loc[row.user_id]
        pred = predict_item_rating_knn(
            user_ratings=user_ratings,
            neighbors=top_neighbors[row.parent_asin]
        )
        predictions.append(pred)

    y_true = ratings_test["rating"].to_numpy(dtype=np.float32)
    y_pred = np.array(predictions, dtype=np.float32)

    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))

    return mse, rmse


def build_top_k_recommendations(filled_matrix, original_matrix, item_titles, top_k=10):
    users = filled_matrix.index.to_numpy()
    items = filled_matrix.columns.to_numpy()
    filled_values = filled_matrix.to_numpy(dtype=np.float32)
    seen_mask = ~np.isnan(original_matrix.to_numpy(dtype=np.float32))

    rows = []
    for u_idx, uid in enumerate(users):
        scores = filled_values[u_idx].copy()
        scores[seen_mask[u_idx]] = -np.inf
        ranked_idx = np.argsort(scores)[::-1][:top_k]

        for rank, item_idx in enumerate(ranked_idx, start=1):
            iid = items[item_idx]
            rows.append({
                "user_id": uid,
                "rank": rank,
                "parent_asin": iid,
                "title": item_titles.loc[iid],
                "pred_rating": float(scores[item_idx])
            })

    return pd.DataFrame(rows)


# Train test split

In [203]:
ratings_eval_train, ratings_eval_test = train_test_split(
    ratings[["user_id", "parent_asin", "rating"]],
    test_size=0.2,
    random_state=RNG
)

train_user_item_matrix = ratings_eval_train.pivot_table(
    index="user_id",
    columns="parent_asin",
    values="rating",
    aggfunc="mean"
)

train_user_item_matrix = train_user_item_matrix.reindex(
    index=user_item_matrix.index,
    columns=user_item_matrix.columns
)

print(f"Train ratings: {len(ratings_eval_train)}")
print(f"Test ratings: {len(ratings_eval_test)}")


Train ratings: 45933
Test ratings: 11484


# Evaluating TF-IDF with item based knn

In [204]:
tfidf_similarity = build_item_similarity_matrix(tfidf_item_embeddings)

tfidf_mse, tfidf_rmse = evaluate_item_knn(
    ratings_test=ratings_eval_test,
    train_matrix=train_user_item_matrix,
    item_similarity=tfidf_similarity,
    top_k=TOP_K
)

display(pd.DataFrame([{
    "embedding": "TF-IDF",
    "MSE": tfidf_mse,
    "RMSE": tfidf_rmse
}]))



,embedding,MSE,RMSE
0,TF-IDF,1.286656,1.134309


# Predicting missing values using Content Based TF-IDF and saving the filled matrix

In [205]:
cb_filled_matrix_tfidf = fill_missing_with_item_knn(
    base_matrix=user_item_matrix,
    item_similarity=tfidf_similarity,
    top_k=TOP_K
)

cb_filled_matrix_tfidf.to_csv("cb_filled_matrix_tfidf.csv", index=True)
display(cb_filled_matrix_tfidf.head())
print("\n statistiche dei ratings (comprende sia quelli gia esistenti che quelli predetti)")
display(cb_filled_matrix_tfidf.stack().describe())

parent_asin,B00Z9TLVK0,B00BJH85SW,B001EYUX4Y,B004BV5O0U,B001ELJEA6,B00B5QGJ5S,B09CG15F86,B001EYUWL8,B00BN5T30E,B000EP3ZLC,...,B00C1ZBFTW,B00NY5ZO0Y,B000P46NKC,B07JYW33NR,B008MAZ0PU,B09GHQVX3N,B000B8J7K0,B0716Q4932,B00XUYWTUC,B0862GHVT9
user_id,,,,,,,,,,,,,,,,,,,,,
AE25ZDXYBK3LHKCZ7XUODANPME4A,4.252619,3.984028,3.969864,3.996642,4.223717,1.000000,4.355153,4.0487,4.249701,3.924225,...,4.628878,4.130668,4.411955,4.456893,3.722383,4.074457,4.554116,4.346506,4.428072,4.338072
AE2A5TMJ6YE6ZNWUAFTC6P5XAHXA,4.252619,3.984028,3.969864,3.996642,4.223717,4.503286,4.355153,4.0487,4.249701,3.924225,...,4.628878,4.130668,4.411955,4.456893,3.722383,4.074457,4.554116,4.346506,4.428072,4.338072
AE2AZ2MNROPF33U6SS53VI22OXJA,4.252619,3.984028,3.969864,3.996642,4.223717,4.503286,4.355153,4.0487,4.249701,3.924225,...,4.628878,4.130668,4.411955,4.456893,3.722383,4.074457,4.554116,4.346506,4.428072,4.338072
AE2CHTOI3XIPJZLQN6DU76LI3VPQ,4.252619,3.984028,3.969864,3.996642,4.223717,4.503286,4.355153,4.0487,4.249701,3.924225,...,4.628878,4.130668,4.411955,4.456893,3.722383,4.074457,4.554116,4.346506,4.428072,4.338072
AE2DHCNGTJRKHBHMYMCMKFDKVS2Q,4.252619,3.984028,3.969864,5.000000,4.223717,4.503286,4.355153,4.0487,4.249701,3.924225,...,4.628878,4.130668,4.411955,4.456893,3.722383,4.074457,4.554116,4.346506,4.428072,4.338072



 statistiche dei ratings (comprende sia quelli gia esistenti che quelli predetti)


count    1.253157e+07
mean     4.250504e+00
std      3.155206e-01
min      9.999999e-01
25%      4.113623e+00
50%      4.265566e+00
75%      4.412941e+00
max      5.000001e+00
dtype: float64

# Top-k recommendations for each users (TF-IDF)

In [206]:
tfidf_top10 = build_top_k_recommendations(
    cb_filled_matrix_tfidf,
    user_item_matrix,
    item_titles,
    top_k=TOP_K
)

display(tfidf_top10.head(30))

,user_id,rank,parent_asin,title,pred_rating
0,AE25ZDXYBK3LHKCZ7XUODANPME4A,1,B01N10NIBP,Nintendo amiibo-Zelda: Breath of the Wild,4.807711
1,AE25ZDXYBK3LHKCZ7XUODANPME4A,2,B095T8C99C,Ratchet & Clank: Rift Apart - PlayStation 5,4.798708
2,AE25ZDXYBK3LHKCZ7XUODANPME4A,3,B0BZFWMYSQ,PlayStation 5 Console (PS5),4.791936
3,AE25ZDXYBK3LHKCZ7XUODANPME4A,4,B075MYT126,Nintendo Switch Pro Controller - Xenoblade Chr...,4.791485
4,AE25ZDXYBK3LHKCZ7XUODANPME4A,5,B0B7TKCZGK,Nintendo Switch™ Pro USB Controller Splatoon™ ...,4.778856
5,AE25ZDXYBK3LHKCZ7XUODANPME4A,6,B01N33LIXR,Nintendo amiibo-Link (Rider): Breath of the Wild,4.777524
6,AE25ZDXYBK3LHKCZ7XUODANPME4A,7,B09DFCB66S,PlayStation 5 Console CFI-1102A,4.775655
7,AE25ZDXYBK3LHKCZ7XUODANPME4A,8,B087NNPYP3,The Legend of Zelda: Breath of the Wild Master...,4.774634
8,AE25ZDXYBK3LHKCZ7XUODANPME4A,9,B08RDC5HL7,Fintie Kids Case for Nintendo Switch Lite 2019...,4.774521
9,AE25ZDXYBK3LHKCZ7XUODANPME4A,10,B08FC5L3RG,PlayStation 5 Console,4.773642


# Evaluating neural embeddings with item based knn

In [207]:
neural_similarity = build_item_similarity_matrix(neural_item_embeddings)

neural_mse, neural_rmse = evaluate_item_knn(
    ratings_test=ratings_eval_test,
    train_matrix=train_user_item_matrix,
    item_similarity=neural_similarity,
    top_k=TOP_K
)

display(pd.DataFrame([{
    "embedding": "Neural",
    "MSE": neural_mse,
    "RMSE": neural_rmse
}]))


,embedding,MSE,RMSE
0,Neural,1.266831,1.125536


# Predicting missing values using neural embeddings and saving the filled matrix

In [208]:
cb_filled_matrix_neural = fill_missing_with_item_knn(
    base_matrix=user_item_matrix,
    item_similarity=neural_similarity,
    top_k=TOP_K
)
cb_filled_matrix_neural.to_csv("cb_filled_matrix_neural.csv", index=True)

display(cb_filled_matrix_neural.head())
print("\n statistiche dei ratings (comprende sia quelli gia esistenti che quelli predetti)")
display(cb_filled_matrix_neural.stack().describe())

parent_asin,B00Z9TLVK0,B00BJH85SW,B001EYUX4Y,B004BV5O0U,B001ELJEA6,B00B5QGJ5S,B09CG15F86,B001EYUWL8,B00BN5T30E,B000EP3ZLC,...,B00C1ZBFTW,B00NY5ZO0Y,B000P46NKC,B07JYW33NR,B008MAZ0PU,B09GHQVX3N,B000B8J7K0,B0716Q4932,B00XUYWTUC,B0862GHVT9
user_id,,,,,,,,,,,,,,,,,,,,,
AE25ZDXYBK3LHKCZ7XUODANPME4A,4.322044,4.309898,3.000000,4.200234,4.259188,4.301713,4.340817,4.000279,4.160041,4.129776,...,4.460098,1.000000,4.419365,4.469667,3.000000,4.231982,4.540268,4.240041,4.509963,4.190151
AE2A5TMJ6YE6ZNWUAFTC6P5XAHXA,4.322044,4.309898,4.089882,4.200234,4.259188,4.301713,4.340817,4.000279,4.160041,4.129776,...,4.460098,4.169962,4.419365,4.469667,3.808708,4.231982,4.540268,4.240041,4.509963,4.190151
AE2AZ2MNROPF33U6SS53VI22OXJA,4.322044,4.309898,4.089882,4.200234,4.259188,4.301713,4.340817,4.000279,4.160041,4.129776,...,4.460098,4.169962,4.419365,4.469667,3.808708,4.231982,4.540268,4.240041,4.509963,4.190151
AE2CHTOI3XIPJZLQN6DU76LI3VPQ,4.322044,4.309898,1.000000,4.200234,4.259188,1.000000,4.340817,4.000279,4.160041,4.129776,...,4.460098,4.169962,4.419365,4.469667,3.808708,4.231982,4.540268,4.240041,4.509963,4.190151
AE2DHCNGTJRKHBHMYMCMKFDKVS2Q,4.322044,4.309898,4.089882,4.200234,5.000000,5.000000,4.340817,4.000279,5.000000,4.129776,...,4.460098,4.169962,4.419365,4.469667,3.808708,4.231982,4.540268,4.240041,4.509963,4.190151



 statistiche dei ratings (comprende sia quelli gia esistenti che quelli predetti)


count    1.253157e+07
mean     4.220075e+00
std      3.142350e-01
min      9.999999e-01
25%      4.090145e+00
50%      4.230040e+00
75%      4.365903e+00
max      5.000000e+00
dtype: float64

# Top-k recommendations for each users (neural)

In [209]:
neural_top10 = build_top_k_recommendations(
    cb_filled_matrix_neural,
    user_item_matrix,
    item_titles,
    top_k=TOP_K
)

display(neural_top10.head(30))

,user_id,rank,parent_asin,title,pred_rating
0,AE25ZDXYBK3LHKCZ7XUODANPME4A,1,B06ZXRH8N7,Nintendo amiibo - New Inkling Squid (Neon Purple),4.830005
1,AE25ZDXYBK3LHKCZ7XUODANPME4A,2,B071CZ2X6L,Nintendo amiibo - Cloud (SSB),4.820004
2,AE25ZDXYBK3LHKCZ7XUODANPME4A,3,B071R6QGRS,Nintendo amiibo - New Inkling Boy (Neon Green),4.810227
3,AE25ZDXYBK3LHKCZ7XUODANPME4A,4,B071D8LQSP,Nintendo amiibo - Pikmin,4.809937
4,AE25ZDXYBK3LHKCZ7XUODANPME4A,5,B00ZOMO3K2,Nintendo NFC Reader/Writer Accessory - Nintend...,4.809617
5,AE25ZDXYBK3LHKCZ7XUODANPME4A,6,B01307Q6ZC,Nintendo NFC Reader/Writer Accessory - Nintend...,4.809617
6,AE25ZDXYBK3LHKCZ7XUODANPME4A,7,B07NQH43DC,Nintendo Amiibo - Ken (Ssbu) - Switch,4.800102
7,AE25ZDXYBK3LHKCZ7XUODANPME4A,8,B07NTTGZTK,Nintendo amiibo - New Inkling Girl (Neon Pink),4.789955
8,AE25ZDXYBK3LHKCZ7XUODANPME4A,9,B07NQLNB4H,Nintendo Amiibo - Young Link (Ssbu) - Switch,4.779813
9,AE25ZDXYBK3LHKCZ7XUODANPME4A,10,B071GPJS1Q,Amiibo - Bowser (Super Mario Odyssey),4.769812


# Frequence based vs Neural: (MSE, RMSE, recommendations)

In [210]:
comparison_cb_embeddings = pd.DataFrame([
    {"embedding": "TF-IDF (basata su frequenza)", "MSE": tfidf_mse, "RMSE": tfidf_rmse},
    {"embedding": "Neural (average_word_embeddings_komninos)", "MSE": neural_mse, "RMSE": neural_rmse}
])
display(comparison_cb_embeddings)

cols = ["user_id", "rank", "title", "pred_rating"]
compare_top10_embeddings = tfidf_top10[cols].merge(
    neural_top10[cols],
    on=["user_id", "rank"],
    suffixes=("_tfidf", "_neural")
).sort_values(["user_id", "rank"])

display(compare_top10_embeddings.head(30))


,embedding,MSE,RMSE
0,TF-IDF (basata su frequenza),1.286656,1.134309
1,Neural (average_word_embeddings_komninos),1.266831,1.125536


,user_id,rank,title_tfidf,pred_rating_tfidf,title_neural,pred_rating_neural
0,AE25ZDXYBK3LHKCZ7XUODANPME4A,1,Nintendo amiibo-Zelda: Breath of the Wild,4.807711,Nintendo amiibo - New Inkling Squid (Neon Purple),4.830005
1,AE25ZDXYBK3LHKCZ7XUODANPME4A,2,Ratchet & Clank: Rift Apart - PlayStation 5,4.798708,Nintendo amiibo - Cloud (SSB),4.820004
2,AE25ZDXYBK3LHKCZ7XUODANPME4A,3,PlayStation 5 Console (PS5),4.791936,Nintendo amiibo - New Inkling Boy (Neon Green),4.810227
3,AE25ZDXYBK3LHKCZ7XUODANPME4A,4,Nintendo Switch Pro Controller - Xenoblade Chr...,4.791485,Nintendo amiibo - Pikmin,4.809937
4,AE25ZDXYBK3LHKCZ7XUODANPME4A,5,Nintendo Switch™ Pro USB Controller Splatoon™ ...,4.778856,Nintendo NFC Reader/Writer Accessory - Nintend...,4.809617
5,AE25ZDXYBK3LHKCZ7XUODANPME4A,6,Nintendo amiibo-Link (Rider): Breath of the Wild,4.777524,Nintendo NFC Reader/Writer Accessory - Nintend...,4.809617
6,AE25ZDXYBK3LHKCZ7XUODANPME4A,7,PlayStation 5 Console CFI-1102A,4.775655,Nintendo Amiibo - Ken (Ssbu) - Switch,4.800102
7,AE25ZDXYBK3LHKCZ7XUODANPME4A,8,The Legend of Zelda: Breath of the Wild Master...,4.774634,Nintendo amiibo - New Inkling Girl (Neon Pink),4.789955
8,AE25ZDXYBK3LHKCZ7XUODANPME4A,9,Fintie Kids Case for Nintendo Switch Lite 2019...,4.774521,Nintendo Amiibo - Young Link (Ssbu) - Switch,4.779813
9,AE25ZDXYBK3LHKCZ7XUODANPME4A,10,PlayStation 5 Console,4.773642,Amiibo - Bowser (Super Mario Odyssey),4.769812
